# 08 — Comparación visual de modelos

Comparación cualitativa de los cinco detectores entrenados sobre D-Fire
(YOLOv8n, YOLO11n, YOLO26s, Faster R-CNN y RT-DETR-L): se toma una imagen
aleatoria del split de test, se corre la inferencia con todos los modelos y se
muestran lado a lado las cajas predichas con su clase y confianza, junto al
ground truth.

Los pesos se leen de las corridas que dejaron los notebooks de entrenamiento
(02 a 06) en Drive (`/content/drive/MyDrive/VCII_DFire/runs`), por lo que el
notebook está pensado para Colab. En local funciona si esas corridas están en
`<repo>/runs`.

In [ ]:
# ============================================================
# Setup general del entorno
# ============================================================

from pathlib import Path
import os
import random
import sys

SEED = 42
random.seed(SEED)

IN_COLAB = "google.colab" in sys.modules
# KAGGLE_KERNEL_RUN_TYPE lo define el runtime de Kaggle y no existe si alguien
# instala el paquete `kaggle` en otra máquina.
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IN_COLAB:
    ENV = "colab"
elif IN_KAGGLE:
    ENV = "kaggle"
else:
    ENV = "local"

print("Entorno:", ENV)
print("Directorio actual:", Path.cwd())

In [ ]:
# ============================================================
# Instalación de dependencias
# ============================================================

REPO_URL = "https://github.com/Gabriela-Sol/VpC2---Deteccion-de-humo-y-fuego"
REPO_NAME = "VpC2---Deteccion-de-humo-y-fuego"
REPO_BRANCH = "main"

RAW_REQUIREMENTS = (
    "https://raw.githubusercontent.com/Gabriela-Sol/"
    f"{REPO_NAME}/{REPO_BRANCH}/requirements.txt"
)

if IN_COLAB:
    !pip install -q -r {RAW_REQUIREMENTS}
elif IN_KAGGLE:
    # Solo ultralytics: instalar requirements.txt completo reemplazaría el
    # torch preinstalado de la imagen (compilado contra su CUDA) por uno
    # cualquiera de PyPI.
    !pip install -q "ultralytics>=8.3,<8.5"

print("Dependencias instaladas.")

In [ ]:
# ============================================================
# Device
# ============================================================
# A diferencia de los notebooks de entrenamiento, acá la CPU es viable: se
# infiere una sola imagen por modelo, así que no se corta si no hay GPU.

import torch

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("GPU:", torch.cuda.get_device_name(0))
else:
    DEVICE = torch.device("cpu")
    print("Sin GPU: la inferencia corre en CPU (más lenta pero viable).")

In [ ]:
# ============================================================
# Almacenamiento (Drive en Colab)
# ============================================================
# Los pesos de las cinco corridas viven en la misma carpeta de Drive donde los
# guardaron los notebooks de entrenamiento.

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/VCII_DFire")
    RUNS_DIR = DRIVE_PROJECT_DIR / "runs"
    WORK_DIR = Path("/content")
elif IN_KAGGLE:
    WORK_DIR = Path("/kaggle/working")
    # En Kaggle no hay Drive: las corridas tienen que copiarse a mano a
    # /kaggle/working/runs (por ejemplo desde un output montado con
    # Add Input > Your Work).
    RUNS_DIR = WORK_DIR / "runs"
else:
    WORK_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    RUNS_DIR = WORK_DIR / "runs"

print("WORK_DIR:", WORK_DIR)
print("Corridas en:", RUNS_DIR)

In [ ]:
# ============================================================
# Clonado o actualización del repositorio
# ============================================================

if ENV == "local":
    # Ya estamos dentro del repo: no hay nada que clonar.
    PROJECT_DIR = WORK_DIR
else:
    PROJECT_DIR = WORK_DIR / REPO_NAME

    if PROJECT_DIR.exists():
        print("El repositorio ya existe. Actualizando...")
        %cd {PROJECT_DIR}
        !git checkout {REPO_BRANCH}
        !git pull origin {REPO_BRANCH}
    else:
        print("Clonando repositorio...")
        %cd {WORK_DIR}
        !git clone -b {REPO_BRANCH} {REPO_URL}.git
        %cd {PROJECT_DIR}

# Necesario para que `import src...` funcione.
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("PROJECT_DIR:", PROJECT_DIR)

In [ ]:
# ============================================================
# Dataset D-Fire (mirror YOLO en Kaggle)
# ============================================================

DATASET_ID = "sayedgamal99/smoke-fire-detection-yolo"


def find_yolo_dataset_dir(root: Path) -> Path:
    for candidate in [root] + [p for p in root.rglob("*") if p.is_dir()]:
        if all(
            (candidate / split / kind).exists()
            for split in ["train", "val"]
            for kind in ["images", "labels"]
        ):
            return candidate
    raise FileNotFoundError("No se encontró una estructura YOLO válida.")


DATA_DIR = None

if IN_KAGGLE:
    # Si el dataset se agregó con Add Input > Datasets ya está montado en
    # /kaggle/input y no hay nada que bajar.
    for mount in sorted(Path("/kaggle/input").glob("*")):
        if not mount.is_dir():
            continue
        try:
            DATA_DIR = find_yolo_dataset_dir(mount)
        except FileNotFoundError:
            continue
        print("Dataset montado desde los inputs del notebook:", mount.name)
        break

if DATA_DIR is None:
    import kagglehub

    DATA_DIR = find_yolo_dataset_dir(Path(kagglehub.dataset_download(DATASET_ID)))

print("Dataset:", DATA_DIR)

In [ ]:
# ============================================================
# Configuración de la comparación
# ============================================================

# Split del que se muestrean las imágenes. `test` no participó ni del
# entrenamiento ni de la selección de best.pt, así que es la comparación más
# honesta entre modelos.
SPLIT = "test"

# Umbral de confianza común a los cinco modelos: sin un umbral explícito cada
# framework aplicaría su default y la comparación visual quedaría sesgada.
CONF_THRESHOLD = 0.25

CLASS_NAMES = {0: "smoke", 1: "fire"}
CLASS_COLORS = {0: "royalblue", 1: "red"}

# Faster R-CNN reserva el 0 para el fondo, así que sus clases vienen corridas
# en 1 respecto de los ids YOLO (ver src/data/yolo_dataset.py).
TORCHVISION_TO_YOLO_LABEL = {1: 0, 2: 1}

# Rutas de pesos tal como las dejaron los notebooks de entrenamiento 02 a 06.
# Los modelos de Ultralytics se cargan desde best.pt; Faster R-CNN solo guarda
# la última época (last.pth), igual que en el notebook 07 de métricas.
WEIGHTS_PATHS = {
    "YOLOv8n": RUNS_DIR / "yolov8n_baseline-2" / "weights" / "best.pt",
    "YOLO11n": RUNS_DIR / "yolo11n_baseline" / "weights" / "best.pt",
    "YOLO26s": RUNS_DIR / "yolo26s_baseline" / "weights" / "best.pt",
    "Faster R-CNN": RUNS_DIR / "fasterrcnn_r50fpn" / "last.pth",
    "RT-DETR-L": RUNS_DIR / "rtdetr_l" / "weights" / "best.pt",
}

faltantes = [name for name, path in WEIGHTS_PATHS.items() if not path.exists()]
if faltantes:
    raise FileNotFoundError(
        f"No se encontraron los pesos de: {', '.join(faltantes)}. "
        f"Verificar que las corridas estén en {RUNS_DIR}."
    )

for name, path in WEIGHTS_PATHS.items():
    print(f"{name}: {path}")

In [ ]:
# ============================================================
# Carga de los cinco modelos
# ============================================================

from ultralytics import RTDETR, YOLO

from src.modeling.detectors import build_fasterrcnn


def load_fasterrcnn(weights_path: Path):
    # `last.pth` es un state_dict plano (ver notebook 04), así que hay que
    # reconstruir la arquitectura con los mismos parámetros del entrenamiento
    # (configs/experiments/fasterrcnn_r50fpn.yaml). `pretrained=False` evita
    # descargar los pesos COCO que el state_dict pisaría de todos modos.
    model = build_fasterrcnn(
        num_classes=3,
        backbone="resnet50_fpn_v2",
        min_size=640,
        max_size=1024,
        pretrained=False,
    )
    model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    return model


MODELS = {
    "YOLOv8n": YOLO(str(WEIGHTS_PATHS["YOLOv8n"])),
    "YOLO11n": YOLO(str(WEIGHTS_PATHS["YOLO11n"])),
    "YOLO26s": YOLO(str(WEIGHTS_PATHS["YOLO26s"])),
    "Faster R-CNN": load_fasterrcnn(WEIGHTS_PATHS["Faster R-CNN"]),
    "RT-DETR-L": RTDETR(str(WEIGHTS_PATHS["RT-DETR-L"])),
}

print("Modelos cargados:", ", ".join(MODELS))

In [ ]:
# ============================================================
# Inferencia unificada y ground truth
# ============================================================
# Todas las salidas se normalizan a una lista de detecciones
# (xyxy en píxeles, id de clase YOLO, confianza) para que el dibujado no tenga
# que saber de qué framework salió cada predicción. En el ground truth la
# confianza es None.

from PIL import Image
from torchvision.transforms.functional import to_tensor


def predict(model, image_path: Path, conf: float = CONF_THRESHOLD):
    """Corre un modelo sobre una imagen y devuelve [(xyxy, cls, score), ...]."""
    if isinstance(model, (YOLO, RTDETR)):
        boxes = model.predict(str(image_path), conf=conf, verbose=False)[0].boxes
        return [
            (tuple(xyxy), int(cls), float(score))
            for xyxy, cls, score in zip(
                boxes.xyxy.cpu().numpy(),
                boxes.cls.cpu().numpy(),
                boxes.conf.cpu().numpy(),
            )
        ]

    # torchvision: espera un tensor float en [0, 1] y normaliza internamente.
    image = to_tensor(Image.open(image_path).convert("RGB")).to(DEVICE)
    with torch.no_grad():
        output = model([image])[0]

    detections = []
    for xyxy, label, score in zip(
        output["boxes"].cpu().numpy(),
        output["labels"].cpu().numpy(),
        output["scores"].cpu().numpy(),
    ):
        if score < conf or int(label) not in TORCHVISION_TO_YOLO_LABEL:
            continue
        detections.append(
            (tuple(xyxy), TORCHVISION_TO_YOLO_LABEL[int(label)], float(score))
        )
    return detections


def load_ground_truth(image_path: Path):
    """Lee la etiqueta YOLO de la imagen y devuelve [(xyxy, cls, None), ...]."""
    label_path = (
        image_path.parent.parent / "labels" / image_path.with_suffix(".txt").name
    )
    with Image.open(image_path) as img:
        width, height = img.size

    detections = []
    if not label_path.exists():
        return detections

    for line in label_path.read_text().splitlines():
        parts = line.split()
        if len(parts) != 5:
            continue
        cls = int(parts[0])
        cx, cy, w, h = map(float, parts[1:])
        box = (
            (cx - w / 2) * width,
            (cy - h / 2) * height,
            (cx + w / 2) * width,
            (cy + h / 2) * height,
        )
        detections.append((box, cls, None))
    return detections

In [ ]:
# ============================================================
# Dibujado de paneles
# ============================================================

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle


def draw_panel(ax, image, detections, title):
    ax.imshow(image)
    ax.set_title(title, fontsize=11)
    ax.axis("off")

    for (x1, y1, x2, y2), cls, score in detections:
        color = CLASS_COLORS.get(cls, "yellow")
        ax.add_patch(
            Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                linewidth=1.5,
                edgecolor=color,
                facecolor="none",
            )
        )
        label = CLASS_NAMES.get(cls, str(cls))
        if score is not None:
            label = f"{label} {score:.2f}"
        ax.text(
            x1,
            max(y1 - 4, 0),
            label,
            fontsize=8,
            color="white",
            bbox={"facecolor": color, "alpha": 0.8, "pad": 1},
        )

## Comparación sobre una imagen aleatoria

Cada ejecución de la celda siguiente muestrea una imagen distinta del split de
test y muestra en una fila el ground truth y las predicciones de los cinco
modelos, con la clase y la confianza de cada caja (humo en azul, fuego en
rojo). El título de cada panel incluye la cantidad de cajas entre paréntesis.

Para repetir exactamente la misma secuencia de imágenes, volver a correr la
celda de setup (que fija `SEED`) y re-ejecutar esta celda desde el principio.

In [ ]:
# ============================================================
# Comparación visual: ground truth + 5 modelos en una fila
# ============================================================
# Celda re-ejecutable: cada corrida muestrea una imagen nueva del split.

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

split_images = sorted(
    p
    for p in (DATA_DIR / SPLIT / "images").iterdir()
    if p.suffix.lower() in IMAGE_EXTENSIONS
)
assert split_images, f"No hay imágenes en {DATA_DIR / SPLIT / 'images'}"

image_path = random.choice(split_images)
image = Image.open(image_path).convert("RGB")

panels = [("Ground truth", load_ground_truth(image_path))]
for name, model in MODELS.items():
    panels.append((name, predict(model, image_path)))

fig, axes = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 4.5))
for ax, (title, detections) in zip(axes, panels):
    draw_panel(ax, image, detections, f"{title} ({len(detections)})")

fig.suptitle(f"{SPLIT}/{image_path.name} — conf >= {CONF_THRESHOLD}", fontsize=13)
fig.tight_layout()
plt.show()